# Import packages

In [1]:
import cv2
from keras.utils import load_img
from keras.saving import load_model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from skimage.measure import regionprops, regionprops_table
from tqdm import trange, tqdm

import segmenteverygrain as seg
import segmenteverygrain.interactions as si
import os

# %matplotlib qt

## Load models

Maybe you need to clone first SAM2. Then add the yaml file in the bottom of the next chunk

In [2]:
# UNET model
unet = load_model(
    "../models/seg_model.keras",
    custom_objects={"weighted_crossentropy": seg.weighted_crossentropy},
)

# Auto-detect device: CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback
import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# SAM 2.1 checkpoints. Download from:
# https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
sam = build_sam2(r"C:\Users\leoni\sam2\sam2\configs\sam2.1\sam2.1_hiera_l.yaml", r"C:\Users\leoni\segmenteverygrain\models\sam2.1_hiera_large.pt", device=device)


## Set your directories

In [13]:
dir = os.chdir("C:/Users/leoni/kirgis_repo/segmenteverygrain/leonieexamples")
dir = os.getcwd()
os.makedirs("ouputgpkg", exist_ok=True)
os.makedirs("inputtiles", exist_ok=True)
os.makedirs("plots", exist_ok=True)


inputtiledir = os.path.join(dir, "inputtiles/")
ouputgpkg = os.path.join(dir, "ouputgpkg/")

plotdir = os.path.join(dir, "plots/")

## Run segmentation

Collect tiles

In [14]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img
import rasterio

# --- collect tiles ---
tiles = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))

print(f"Found {len(tiles)} tiles")


if len(tiles) == 0:
    raise RuntimeError("No tiles found.")

# --- read pixel size once from first tile ---
with rasterio.open(tiles[0]) as src:
    xres, yres = src.res          # CRS units per pixel (usually meters)
    crs = src.crs

gsd_units = float((xres + yres) / 2.0)
gsd_mm = gsd_units * 1000.0       # assumes CRS units are meters (e.g., UTM)

minor_mm = 20.0                   # 2 cm
minor_px = minor_mm / gsd_mm

# area threshold from minor-axis (circle assumption)
min_area_px = float(np.pi * (minor_px / 2.0) ** 2)

print("CRS:", crs)
print(f"Pixel size: {gsd_units:.6f} units/px (~{gsd_mm:.3f} mm/px)")
print(f"2 cm => minor_px={minor_px:.2f} px -> min_area_px={min_area_px:.1f} px^2")



# --- loop ---
for i, fname in enumerate(tiles, start=1):
    tile_id = os.path.splitext(os.path.basename(fname))[0]
    print(f"[{i}/{len(tiles)}] {tile_id}")

    # load + predict
    image = np.array(load_img(fname))
    image_pred = seg.predict_image(image, unet, I=256)

    # prompts (Unet -> points)
    labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=10.0)

    # SAM segmentation
    all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
        sam,
        image,
        image_pred,
        coords,
        labels,
        min_area=min_area_px,
        plot_image=True,
        remove_edge_grains=True,
        remove_large_objects=True,
    )
    figname =os.path.join(plotdir, f"{tile_id}.png")

    fig.savefig(figname, dpi = 200, bbox_inches="tight")
    plt.close(fig)
    print("Saved the Plot")
    print("done with segmentation")






Found 1 tiles
CRS: EPSG:32643
Pixel size: 0.003501 units/px (~3.501 mm/px)
2 cm => minor_px=5.71 px -> min_area_px=25.6 px^2
[1/1] tile_r000008_c000026
segmenting image tiles...


100%|██████████| 3/3 [00:02<00:00,  1.24it/s]


creating masks using SAM...


100%|██████████| 822/822 [01:33<00:00,  8.80it/s]


finding overlapping polygons...


...

finding best polygons...


100%|██████████| 200/200 [00:32<00:00,  6.17it/s]


creating labeled image...


100%|██████████| 244/244 [00:01<00:00, 128.43it/s]


Saved the Plot
done with segmentation


In [10]:
min_area_px

25.62514189487139